Proecessing CSV and Excel files:

In [2]:
import pandas as pd
import os

In [3]:
os.makedirs("data/structured_files", exist_ok = True)

In [4]:
data = {
    "Product_ID": [101, 102, 103, 104, 105, 106, 107, 108, 109, 110],
    "Product_Name": [
        "Wireless Mouse",
        "Mechanical Keyboard",
        "Gaming Monitor",
        "USB-C Hub",
        "Laptop Stand",
        "Noise Cancelling Headphones",
        "External SSD",
        "Webcam",
        "Bluetooth Speaker",
        "Smart Watch"
    ],
    "Category": [
        "Accessories",
        "Accessories",
        "Monitors",
        "Accessories",
        "Furniture",
        "Audio",
        "Storage",
        "Cameras",
        "Audio",
        "Wearables"
    ],
    "Brand": [
        "LogiTech",
        "KeyChron",
        "Samsung",
        "Anker",
        "AmazonBasics",
        "Sony",
        "SanDisk",
        "LogiTech",
        "JBL",
        "Apple"
    ],
    "Price": [
        799,
        6499,
        18999,
        2499,
        1999,
        12999,
        7999,
        3999,
        4999,
        29999
    ],
    "Stock": [150, 80, 35, 120, 60, 25, 45, 70, 90, 30],
    "Rating": [4.5, 4.8, 4.7, 4.4, 4.3, 4.8, 4.6, 4.2, 4.5, 4.9],
    "Discount (%)": [10, 15, 8, 20, 5, 12, 18, 10, 25, 7],
    "Supplier": [
        "TechWorld",
        "KeyStore",
        "DisplayHub",
        "Anker India",
        "OfficePro",
        "Sony India",
        "StorageMart",
        "CameraZone",
        "AudioPlus",
        "Apple India"
    ]
}

df = pd.DataFrame(data)

df.to_csv("data/structured_files/data.csv", index = False)

In [5]:
with pd.ExcelWriter("data/structured_files/inventory.xlsx") as writer:
    df.to_excel(writer, sheet_name = "Products", index = False)

    summary = {
        "Category" : ["Electronics", "Accessories"],
        "Total_Items" : [28,32],
        "Total_Value" : [50000,25000]
    }

    pd.DataFrame(summary).to_excel(writer, sheet_name = "Summary", index = False)

Processing CSV File:

In [6]:
from langchain_community.document_loaders import CSVLoader
from langchain_community.document_loaders import UnstructuredCSVLoader

try:

    csv_loader = CSVLoader(
        file_path = 'data/structured_files/data.csv',
        encoding = 'utf-8',
        csv_args = {
            'delimiter' : ',',
            'quotechar' : '"',
        }
    )

    csv_docs = csv_loader.load() # Row wise documents
    print(f"Total Documents: {len(csv_docs)}") # one per row
    print(f"Metadata of doc1: {csv_docs[0].metadata}")
    print(f"Preview of doc1: {csv_docs[0].page_content}")

except Exception as e:
    print(f"Error: {e}")

/tmp/ipykernel_41831/1660281119.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import CSVLoader
/home/jaywardhan/RAG_Udemy/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total Documents: 10
Metadata of doc1: {'source': 'data/structured_files/data.csv', 'row': 0}
Preview of doc1: Product_ID: 101
Product_Name: Wireless Mouse
Category: Accessories
Brand: LogiTech
Price: 799
Stock: 150
Rating: 4.5
Discount (%): 10
Supplier: TechWorld


Custom CSV Processing for adding more metadata:

In [7]:
from typing import List
from langchain_core.documents import Document

def customCSV(filepath: str) -> List[Document]:
    df = pd.read_csv(filepath)

    ## Adding precise Content:
    document = []

    for idx , row in df.iterrows():

        content = f"""Product Information:
            Name: {row['Product_Name']}
            Category: {row['Category']}
            Brand: {row['Brand']}
            Price: ${row['Price']}
            Stock: {row['Stock']}
            Rating: {row['Rating']}
            Discount: {row['Discount (%)']}%
            Supplier: {row['Supplier']}

        """

        doc = Document (
            page_content = content,
            metadata = {
                'source' : filepath,
                'row_idx' : idx,
                'Name' : row['Product_Name'],
                'Category': row['Category'],
                ' Brand' : row['Brand'],
                ' Price' : row['Price'],
                ' Stock' : row['Stock'],
                ' Rating' : row['Rating'],
                ' Discount' : row['Discount (%)'],
                ' Supplier' : row['Supplier'],
            }
        )

        document.append(doc)

    return document

In [8]:
docs = customCSV("data/structured_files/data.csv")

In [9]:
docs

[Document(metadata={'source': 'data/structured_files/data.csv', 'row_idx': 0, 'Name': 'Wireless Mouse', 'Category': 'Accessories', ' Brand': 'LogiTech', ' Price': 799, ' Stock': 150, ' Rating': 4.5, ' Discount': 10, ' Supplier': 'TechWorld'}, page_content='Product Information:\n            Name: Wireless Mouse\n            Category: Accessories\n            Brand: LogiTech\n            Price: $799\n            Stock: 150\n            Rating: 4.5\n            Discount: 10%\n            Supplier: TechWorld\n\n        '),
 Document(metadata={'source': 'data/structured_files/data.csv', 'row_idx': 1, 'Name': 'Mechanical Keyboard', 'Category': 'Accessories', ' Brand': 'KeyChron', ' Price': 6499, ' Stock': 80, ' Rating': 4.8, ' Discount': 15, ' Supplier': 'KeyStore'}, page_content='Product Information:\n            Name: Mechanical Keyboard\n            Category: Accessories\n            Brand: KeyChron\n            Price: $6499\n            Stock: 80\n            Rating: 4.8\n            Dis

Processing Excel file

In [10]:
from langchain_community.document_loaders import UnstructuredExcelLoader

try:

    excel_loader = UnstructuredExcelLoader(
        file_path = "data/structured_files/inventory.xlsx",
        mode = "elements"
    )

    excel_docs = excel_loader.load()
    print(excel_docs)

except Exception as e:
    print(f"Error: {e}")

[Document(metadata={'source': 'data/structured_files/inventory.xlsx', 'file_directory': 'data/structured_files', 'filename': 'inventory.xlsx', 'last_modified': '2026-08-08T23:59:03', 'page_name': 'Products', 'page_number': 1, 'text_as_html': '<table><tr><td>Product_ID</td><td>Product_Name</td><td>Category</td><td>Brand</td><td>Price</td><td>Stock</td><td>Rating</td><td>Discount (%)</td><td>Supplier</td></tr><tr><td>101</td><td>Wireless Mouse</td><td>Accessories</td><td>LogiTech</td><td>799</td><td>150</td><td>4.5</td><td>10</td><td>TechWorld</td></tr><tr><td>102</td><td>Mechanical Keyboard</td><td>Accessories</td><td>KeyChron</td><td>6499</td><td>80</td><td>4.8</td><td>15</td><td>KeyStore</td></tr><tr><td>103</td><td>Gaming Monitor</td><td>Monitors</td><td>Samsung</td><td>18999</td><td>35</td><td>4.7</td><td>8</td><td>DisplayHub</td></tr><tr><td>104</td><td>USB-C Hub</td><td>Accessories</td><td>Anker</td><td>2499</td><td>120</td><td>4.4</td><td>20</td><td>Anker India</td></tr><tr><td>1

In [11]:
print(len(excel_docs))

2


In [12]:
print(excel_docs[1])

page_content='Category Total_Items Total_Value Electronics 28 50000 Accessories 32 25000' metadata={'source': 'data/structured_files/inventory.xlsx', 'file_directory': 'data/structured_files', 'filename': 'inventory.xlsx', 'last_modified': '2026-08-08T23:59:03', 'page_name': 'Summary', 'page_number': 2, 'text_as_html': '<table><tr><td>Category</td><td>Total_Items</td><td>Total_Value</td></tr><tr><td>Electronics</td><td>28</td><td>50000</td></tr><tr><td>Accessories</td><td>32</td><td>25000</td></tr></table>', 'languages': ['eng'], 'filetype': 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet', 'category': 'Table', 'element_id': 'aca3e5ef394d0f0c28f9589f50bd60b6'}
